# TPU Test 1: Text generation on vLLM-TPU
Smoke test: load Qwen3-4B-Thinking-2507 via vLLM-TPU, generate 20 prompts (4 proposals each).
Metrics: tokens/s, output sanity, any errors.
Report: success + (tokens/s, output samples), or exact blocker if failed.

In [ ]:
import os, json, time
print("TPU Test 1: vLLM-TPU text generation")
print()
print("Check TPU availability...")
try:
    import jax
    print(f"JAX version: {jax.__version__}")
    print(f"JAX devices: {len(jax.devices())} available")
except ImportError:
    print("JAX not available; install will be attempted by vLLM")

In [ ]:
# Install and load vLLM-TPU
print("\nLoading vLLM-TPU...")
try:
    from vllm import LLM, SamplingParams
    print("vLLM already installed")
except ImportError:
    print("Installing vLLM[tpu]...")
    os.system("pip install -q vllm[tpu]")
    from vllm import LLM, SamplingParams

In [ ]:
# Load model
print("\nLoading Qwen3-4B-Thinking-2507 on TPU v5e-8...")
model_id = "Qwen/Qwen3-4B-Thinking-2507"
error_report = None

try:
    llm = LLM(
        model=model_id,
        tensor_parallel_size=8,
        dtype="bfloat16",
    )
    print(f"SUCCESS: Loaded {model_id}")
except Exception as e:
    error_report = f"FAILED: {type(e).__name__}: {str(e)[:400]}"
    print(error_report)
    
if error_report:
    import sys
    print(f"\nReport: vLLM-TPU text generation FAILED")
    print(f"Blocker: {error_report}")
    sys.exit(1)

In [ ]:
# Generate on 20 sample prompts
import time

print("\nGenerating 20 prompts (4 proposals each)...")
prompts = [
    "Fix the bug: def add(a, b): return a + c",
    "Next line in code?",
    "Complete function:",
] * 7
prompts = prompts[:20]  # Use first 20

sampling_params = SamplingParams(
    max_tokens=12000,
    temperature=0.7,
    top_p=0.9,
    n=4,
)

error_report = None
t0 = time.time()
try:
    outputs = llm.generate(prompts, sampling_params)
    elapsed = time.time() - t0
    total_tokens = sum(len(o.outputs[0].token_ids) if o.outputs else 0 for o in outputs)
    tokens_per_sec = total_tokens / elapsed if elapsed > 0 else 0
    print(f"Generation completed in {elapsed:.1f} s")
    print(f"Tokens/s: {tokens_per_sec:.1f}")
except Exception as e:
    error_report = f"FAILED: {type(e).__name__}: {str(e)[:400]}"
    print(error_report)
    tokens_per_sec = 0

if error_report:
    import sys
    print(f"\nReport: vLLM-TPU text generation FAILED")
    print(f"Blocker: {error_report}")
    sys.exit(1)

In [ ]:
# Sanity check: sample outputs
print("\nSample outputs (first 2 prompts):")
for i, output in enumerate(outputs[:2]):
    if output.outputs:
        text = output.outputs[0].text[:150]
        print(f"  Prompt {i}: {text}...")

print("\n" + "="*60)
print("Report: vLLM-TPU text generation PASSED")
print(f"  Model: {model_id}")
print(f"  Platform: TPU v5e-8")
print(f"  Elapsed: {elapsed:.1f} s")
print(f"  Tokens/s: {tokens_per_sec:.1f}")
print(f"  Total tokens: {total_tokens}")
print("="*60)